[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/09-imputacion.ipynb)


# Machine Learning: Imputación de Datos Faltantes

## Métodos, métricas, comparación y buenas prácticas

### Objetivo

Aprender a detectar, entender, imputar y evaluar valores faltantes en distintos tipos de datos.

Trabajaremos con:

- variables numéricas;
- variables categóricas;
- variables booleanas;
- fechas;
- series temporales.

### Métodos incluidos

#### Numéricos
- Media
- Mediana
- Constante
- Mediana por grupo
- KNN Imputer
- Iterative Imputer / MICE
- Interpolación

#### Categóricos
- Moda
- Categoría `"DESCONOCIDO"`
- Imputación predictiva con Random Forest

#### Booleanos
- Moda
- Valor constante

#### Fechas y series temporales
- Mediana temporal
- Forward fill
- Backward fill
- Interpolación

### Evaluación
- MAE
- RMSE
- NRMSE
- Accuracy de imputación
- F1 macro
- Error absoluto en días
- Impacto en un modelo de Machine Learning
- Validación cruzada
- Pipelines para evitar data leakage

> **Idea central:** imputar no significa simplemente llenar `NaN`. Debemos justificar el método y evaluar el impacto de la imputación.



# 1. ¿Qué es imputar?

Imputar significa reemplazar un valor faltante por un valor estimado.

Ejemplo:

| Edad | Ingreso |
|---:|---:|
| 28 | 3.200.000 |
| 42 | NaN |
| 35 | 4.100.000 |

El valor faltante podría estimarse mediante:

- media;
- mediana;
- vecinos similares;
- relaciones con otras variables;
- un modelo predictivo.

La imputación **no recupera necesariamente el valor verdadero**. Produce una estimación bajo ciertos supuestos.



# 2. ¿Eliminar o imputar?

Eliminar observaciones puede ser razonable cuando:

- son muy pocas;
- la pérdida parece aleatoria;
- disponemos de suficiente información restante.

Puede ser problemático cuando:

- desaparece una parte importante del dataset;
- los faltantes se concentran en ciertos grupos;
- perdemos información importante;
- introducimos sesgo.

Antes de decidir debemos estudiar:

1. cuánto falta;
2. dónde falta;
3. por qué podría faltar.



# 3. MCAR, MAR y MNAR

## MCAR — Missing Completely At Random

La ausencia no depende de variables observadas ni del propio valor.

Ejemplo: un sensor falla aleatoriamente.

## MAR — Missing At Random

La ausencia está relacionada con otras variables observadas.

Ejemplo: el ingreso se reporta menos entre ciertos grupos de edad.

## MNAR — Missing Not At Random

La ausencia está relacionada con el propio valor faltante o con información no observada.

Ejemplo: personas con ingresos muy altos podrían evitar reportarlos.

> En datos reales, diferenciar MAR de MNAR puede ser difícil. No debemos asumir automáticamente que los faltantes son aleatorios.



# 4. Tipo de variable y métodos frecuentes

| Tipo | Métodos candidatos |
|---|---|
| Numérica simétrica | Media, KNN, Iterative |
| Numérica sesgada/outliers | Mediana, KNN |
| Categórica nominal | Moda, DESCONOCIDO, modelo |
| Categórica ordinal | Moda o método que respete el orden |
| Booleana | Moda o constante |
| Fecha | Mediana temporal, ffill, bfill |
| Serie temporal | Interpolación, ffill, modelos temporales |
| Variable objetivo | Normalmente no se imputa |

La variable objetivo `y` merece especial cuidado porque imputarla puede crear etiquetas artificiales.



# 5. Dataset didáctico mixto

Crearemos un dataset sintético de clientes con 4.000 observaciones.

La ventaja es que conoceremos el dataset completo **antes de ocultar valores**.

Así podremos preguntar:

> ¿Qué tan cerca quedó la imputación del valor verdadero?


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

n = 4000

edad = np.clip(
    rng.normal(39, 11, n),
    18,
    75
).round()

ciudad = rng.choice(
    ["Bogotá", "Medellín", "Cali", "Barranquilla"],
    size=n,
    p=[0.45, 0.22, 0.20, 0.13]
)

segmento = rng.choice(
    ["Básico", "Premium", "Empresarial"],
    size=n,
    p=[0.55, 0.30, 0.15]
)

antiguedad = np.clip(
    rng.gamma(3, 2.2, n),
    0.1,
    25
)

ajuste_segmento = pd.Series(segmento).map({
    "Básico": 0,
    "Premium": 1_800_000,
    "Empresarial": 4_000_000
}).to_numpy()

ingreso = np.maximum(
    1_200_000
    + edad * 55_000
    + antiguedad * 160_000
    + ajuste_segmento
    + rng.normal(0, 1_100_000, n),
    800_000
)

compras = rng.poisson(
    np.clip(
        2
        + antiguedad / 2
        + (segmento == "Premium") * 2
        + (segmento == "Empresarial") * 4,
        0.5,
        None
    )
)

satisfaccion = np.clip(
    rng.normal(
        3.5
        + (segmento == "Premium") * 0.25
        + (segmento == "Empresarial") * 0.35,
        0.7,
        n
    ),
    1,
    5
)

canal = rng.choice(
    ["Web", "Tienda", "App", "Call Center"],
    size=n,
    p=[0.35, 0.25, 0.30, 0.10]
)

activo = rng.choice(
    [True, False],
    size=n,
    p=[0.84, 0.16]
)

fecha_registro = (
    pd.Timestamp("2018-01-01")
    + pd.to_timedelta(
        rng.integers(
            0,
            365 * 8,
            size=n
        ),
        unit="D"
    )
)

logit = (
    -1.8
    + 0.045 * (edad - 40)
    - 0.18 * antiguedad
    - 0.35 * satisfaccion
    + 0.55 * (segmento == "Básico")
    + 0.45 * (canal == "Call Center")
    + rng.normal(0, 0.7, n)
)

prob_churn = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, prob_churn)

df_completo = pd.DataFrame({
    "edad": edad.astype(float),
    "ingreso": ingreso,
    "antiguedad": antiguedad,
    "compras": compras.astype(float),
    "satisfaccion": satisfaccion,
    "ciudad": ciudad,
    "segmento": segmento,
    "canal": canal,
    "activo": activo,
    "fecha_registro": fecha_registro,
    "churn": churn
})

display(df_completo.head())
print("Dimensiones:", df_completo.shape)



# 6. Introducir faltantes controlados

Crearemos ejemplos de:

- MCAR;
- MAR;
- MNAR didáctico.

El dataset `df_completo` permanecerá intacto como referencia.


In [ ]:

df_missing = df_completo.copy()

# MCAR
mask_edad = rng.random(n) < 0.12
df_missing.loc[mask_edad, "edad"] = np.nan

mask_satisfaccion = rng.random(n) < 0.10
df_missing.loc[mask_satisfaccion, "satisfaccion"] = np.nan

mask_ciudad = rng.random(n) < 0.08
df_missing.loc[mask_ciudad, "ciudad"] = np.nan

mask_activo = rng.random(n) < 0.07
df_missing.loc[mask_activo, "activo"] = np.nan

mask_fecha = rng.random(n) < 0.06
df_missing.loc[mask_fecha, "fecha_registro"] = pd.NaT

# MAR: ingreso falta más entre menores de 30
prob_ingreso = np.where(
    df_completo["edad"] < 30,
    0.28,
    0.08
)
mask_ingreso = rng.random(n) < prob_ingreso
df_missing.loc[mask_ingreso, "ingreso"] = np.nan

# MAR: segmento falta más en Call Center
prob_segmento = np.where(
    df_completo["canal"] == "Call Center",
    0.20,
    0.06
)
mask_segmento = rng.random(n) < prob_segmento
df_missing.loc[mask_segmento, "segmento"] = np.nan

# MNAR didáctico: compras altas tienen más probabilidad de faltar
prob_compras = np.where(
    df_completo["compras"] >= 8,
    0.22,
    0.05
)
mask_compras = rng.random(n) < prob_compras
df_missing.loc[mask_compras, "compras"] = np.nan

display(df_missing.head())


# 7. Diagnóstico de faltantes

In [ ]:

faltantes = pd.DataFrame({
    "Cantidad": df_missing.isna().sum(),
    "Porcentaje_%": df_missing.isna().mean() * 100
}).sort_values(
    "Porcentaje_%",
    ascending=False
)

display(faltantes.round(2))


In [ ]:

faltantes_plot = faltantes[
    faltantes["Cantidad"] > 0
]

plt.figure(figsize=(10,5))
plt.bar(
    faltantes_plot.index,
    faltantes_plot["Porcentaje_%"]
)
plt.ylabel("Faltantes (%)")
plt.title("Porcentaje de valores faltantes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

matriz_missing = (
    df_missing
    .isna()
    .astype(int)
)

plt.figure(figsize=(12,7))
plt.imshow(
    matriz_missing.T,
    aspect="auto",
    interpolation="nearest"
)
plt.yticks(
    range(len(matriz_missing.columns)),
    matriz_missing.columns
)
plt.xlabel("Observaciones")
plt.ylabel("Variables")
plt.title("Mapa de valores faltantes")
plt.colorbar(label="1 = faltante")
plt.tight_layout()
plt.show()



# 8. Missing Indicator

El hecho de que una variable falte puede contener información.

Podemos crear una columna adicional:

`ingreso_missing = 1`

Esto puede ser especialmente útil en patrones MAR o MNAR.


In [ ]:

df_indicadores = df_missing.copy()

for col in [
    "edad",
    "ingreso",
    "compras",
    "satisfaccion",
    "ciudad",
    "segmento"
]:
    df_indicadores[
        f"{col}_missing"
    ] = (
        df_indicadores[col]
        .isna()
        .astype(int)
    )

display(
    df_indicadores[
        [
            "ingreso",
            "ingreso_missing",
            "segmento",
            "segmento_missing"
        ]
    ].head(10)
)


# 9. Variables numéricas

In [ ]:

variables_numericas = [
    "edad",
    "ingreso",
    "antiguedad",
    "compras",
    "satisfaccion"
]



# 10. Media

La media puede funcionar bien cuando:

- la distribución es aproximadamente simétrica;
- no existen outliers fuertes.

Es sensible a valores extremos.


In [ ]:

from sklearn.impute import SimpleImputer

imputer_media = SimpleImputer(
    strategy="mean"
)

df_media = pd.DataFrame(
    imputer_media.fit_transform(
        df_missing[variables_numericas]
    ),
    columns=variables_numericas,
    index=df_missing.index
)

display(df_media.head())



# 11. Mediana

La mediana es más robusta frente a:

- outliers;
- distribuciones asimétricas.


In [ ]:

imputer_mediana = SimpleImputer(
    strategy="median"
)

df_mediana = pd.DataFrame(
    imputer_mediana.fit_transform(
        df_missing[variables_numericas]
    ),
    columns=variables_numericas,
    index=df_missing.index
)


# 12. Constante

In [ ]:

imputer_constante = SimpleImputer(
    strategy="constant",
    fill_value=0
)

df_constante = pd.DataFrame(
    imputer_constante.fit_transform(
        df_missing[variables_numericas]
    ),
    columns=variables_numericas,
    index=df_missing.index
)



Imputar 0 solo es adecuado cuando 0 tiene un significado válido o queremos representar explícitamente ausencia.

No debe utilizarse automáticamente.


# 13. Imputación por grupo

In [ ]:

df_grupo = df_missing.copy()

mediana_grupo = (
    df_grupo
    .groupby("segmento")["ingreso"]
    .transform("median")
)

df_grupo["ingreso"] = (
    df_grupo["ingreso"]
    .fillna(mediana_grupo)
    .fillna(
        df_missing["ingreso"]
        .median()
    )
)

print(
    "Faltantes en ingreso:",
    df_grupo["ingreso"].isna().sum()
)



# 14. KNN Imputer

KNN utiliza observaciones similares.

Como utiliza distancias, el escalamiento es importante.


In [ ]:

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

X_num_missing = (
    df_missing[
        variables_numericas
    ].copy()
)

scaler_knn = StandardScaler()

X_num_scaled = scaler_knn.fit_transform(
    X_num_missing
)

knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

X_knn_scaled = knn_imputer.fit_transform(
    X_num_scaled
)

X_knn = scaler_knn.inverse_transform(
    X_knn_scaled
)

df_knn = pd.DataFrame(
    X_knn,
    columns=variables_numericas,
    index=df_missing.index
)

display(df_knn.head())



# 15. Iterative Imputer / MICE

Iterative Imputer estima cada variable incompleta usando las demás.

Proceso:

1. relleno inicial;
2. selecciona una variable;
3. la predice usando las otras;
4. actualiza;
5. repite.

Puede capturar relaciones multivariadas.


In [ ]:

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=15,
    random_state=RANDOM_STATE,
    initial_strategy="median"
)

df_iterative = pd.DataFrame(
    iterative_imputer.fit_transform(
        df_missing[
            variables_numericas
        ]
    ),
    columns=variables_numericas,
    index=df_missing.index
)

display(df_iterative.head())



# 16. Métricas numéricas

Evaluaremos únicamente las posiciones que ocultamos.

## MAE
Error absoluto promedio.

## RMSE
Penaliza errores grandes.

## NRMSE
RMSE dividido por la desviación estándar del verdadero valor.

### Regla
Menor es mejor.


In [ ]:

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error
)

def metricas_imputacion_numerica(
    verdadero,
    imputado
):
    verdadero = np.asarray(verdadero)
    imputado = np.asarray(imputado)

    mae = mean_absolute_error(
        verdadero,
        imputado
    )

    rmse = root_mean_squared_error(
        verdadero,
        imputado
    )

    std = np.std(
        verdadero,
        ddof=0
    )

    nrmse = (
        rmse / std
        if std > 0
        else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "NRMSE": nrmse
    }


# 17. Comparación de imputadores numéricos

In [ ]:

mascaras_numericas = {
    "edad": mask_edad,
    "ingreso": mask_ingreso,
    "compras": mask_compras,
    "satisfaccion": mask_satisfaccion
}

metodos_numericos = {
    "Media": df_media,
    "Mediana": df_mediana,
    "Constante_0": df_constante,
    "KNN": df_knn,
    "Iterative": df_iterative
}

resultados_num = []

for variable, mascara in mascaras_numericas.items():

    verdadero = (
        df_completo.loc[
            mascara,
            variable
        ]
    )

    for metodo, df_imp in metodos_numericos.items():

        imputado = (
            df_imp.loc[
                mascara,
                variable
            ]
        )

        m = metricas_imputacion_numerica(
            verdadero,
            imputado
        )

        resultados_num.append({
            "Variable": variable,
            "Método": metodo,
            **m
        })

tabla_metricas_num = pd.DataFrame(
    resultados_num
)

display(
    tabla_metricas_num
    .sort_values(
        ["Variable", "NRMSE"]
    )
    .round(4)
)


# 18. Ranking promedio numérico

In [ ]:

ranking_numerico = (
    tabla_metricas_num
    .groupby("Método")["NRMSE"]
    .mean()
    .sort_values()
    .to_frame("NRMSE_promedio")
)

display(
    ranking_numerico.round(4)
)


In [ ]:

plt.figure(figsize=(8,5))
plt.bar(
    ranking_numerico.index,
    ranking_numerico["NRMSE_promedio"]
)
plt.ylabel("NRMSE promedio")
plt.title("Comparación global de imputadores numéricos")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# 19. Distribución antes y después de imputar

In [ ]:

variable = "ingreso"

plt.figure(figsize=(10,6))

plt.hist(
    df_completo[variable],
    bins=35,
    alpha=0.45,
    label="Original"
)

plt.hist(
    df_mediana[variable],
    bins=35,
    alpha=0.45,
    label="Mediana"
)

plt.hist(
    df_knn[variable],
    bins=35,
    alpha=0.45,
    label="KNN"
)

plt.xlabel(variable)
plt.ylabel("Frecuencia")
plt.title("Distribución original vs imputada")
plt.legend()
plt.show()



# 20. Variables categóricas: moda


In [ ]:

variables_categoricas = [
    "ciudad",
    "segmento",
    "canal"
]

imputer_moda = SimpleImputer(
    strategy="most_frequent"
)

df_cat_moda = pd.DataFrame(
    imputer_moda.fit_transform(
        df_missing[
            variables_categoricas
        ]
    ),
    columns=variables_categoricas,
    index=df_missing.index
)

display(df_cat_moda.head())



# 21. Categoría explícita `"DESCONOCIDO"`

En algunos problemas conviene conservar la ausencia como categoría propia.


In [ ]:

imputer_unknown = SimpleImputer(
    strategy="constant",
    fill_value="DESCONOCIDO"
)

df_cat_unknown = pd.DataFrame(
    imputer_unknown.fit_transform(
        df_missing[
            variables_categoricas
        ]
    ),
    columns=variables_categoricas,
    index=df_missing.index
)



# 22. Métricas categóricas

## Accuracy
Porcentaje de categorías imputadas correctamente.

## F1 macro
Promedia F1 entre categorías.

Mayor es mejor.


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

def metricas_categoricas(
    verdadero,
    imputado
):
    return {
        "Accuracy": accuracy_score(
            verdadero,
            imputado
        ),
        "F1_macro": f1_score(
            verdadero,
            imputado,
            average="macro",
            zero_division=0
        )
    }


In [ ]:

resultados_cat = []

for variable, mascara in {
    "ciudad": mask_ciudad,
    "segmento": mask_segmento
}.items():

    verdadero = (
        df_completo.loc[
            mascara,
            variable
        ]
    )

    for metodo, df_imp in {
        "Moda": df_cat_moda,
        "DESCONOCIDO": df_cat_unknown
    }.items():

        m = metricas_categoricas(
            verdadero,
            df_imp.loc[
                mascara,
                variable
            ]
        )

        resultados_cat.append({
            "Variable": variable,
            "Método": metodo,
            **m
        })

tabla_cat = pd.DataFrame(
    resultados_cat
)

display(tabla_cat.round(4))



# 23. Imputación categórica predictiva

Podemos entrenar un clasificador para estimar la categoría faltante.

Ejemplo:

> predecir `segmento` utilizando otras variables.


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

pred_num = [
    "edad",
    "ingreso",
    "antiguedad",
    "compras",
    "satisfaccion"
]

pred_cat = [
    "ciudad",
    "canal"
]

prep_segmento = ColumnTransformer([
    (
        "num",
        SimpleImputer(
            strategy="median"
        ),
        pred_num
    ),
    (
        "cat",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        pred_cat
    )
])

modelo_segmento = Pipeline([
    (
        "prep",
        prep_segmento
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=150,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

observados = (
    df_missing["segmento"]
    .notna()
)

modelo_segmento.fit(
    df_missing.loc[
        observados,
        pred_num + pred_cat
    ],
    df_missing.loc[
        observados,
        "segmento"
    ]
)

faltantes = (
    df_missing["segmento"]
    .isna()
)

pred_segmento = modelo_segmento.predict(
    df_missing.loc[
        faltantes,
        pred_num + pred_cat
    ]
)

segmento_modelo = (
    df_missing["segmento"]
    .copy()
)

segmento_modelo.loc[
    faltantes
] = pred_segmento


In [ ]:

verdadero_segmento = (
    df_completo.loc[
        mask_segmento,
        "segmento"
    ]
)

comparacion_segmento = pd.DataFrame({
    "Moda": metricas_categoricas(
        verdadero_segmento,
        df_cat_moda.loc[
            mask_segmento,
            "segmento"
        ]
    ),
    "Random Forest": metricas_categoricas(
        verdadero_segmento,
        segmento_modelo.loc[
            mask_segmento
        ]
    )
}).T

display(
    comparacion_segmento.round(4)
)


# 24. Variables booleanas

In [ ]:

imputer_bool = SimpleImputer(
    strategy="most_frequent"
)

activo_imputado = (
    imputer_bool
    .fit_transform(
        df_missing[["activo"]]
    )
    .ravel()
)

activo_imputado = pd.Series(
    activo_imputado,
    index=df_missing.index
)

accuracy_bool = accuracy_score(
    df_completo.loc[
        mask_activo,
        "activo"
    ].astype(bool),
    activo_imputado.loc[
        mask_activo
    ].astype(bool)
)

print(
    "Accuracy de imputación:",
    round(accuracy_bool, 4)
)



# 25. Fechas: mediana temporal

En registros independientes podemos utilizar una fecha mediana como baseline.


In [ ]:

fechas_validas = (
    df_missing["fecha_registro"]
    .dropna()
)

mediana_ns = int(
    fechas_validas
    .astype("int64")
    .median()
)

mediana_fecha = pd.to_datetime(
    mediana_ns
)

fecha_imputada = (
    df_missing["fecha_registro"]
    .fillna(
        mediana_fecha
    )
)

print(
    "Fecha mediana:",
    mediana_fecha.date()
)


# 26. Error de imputación en días

In [ ]:

error_dias = (
    (
        fecha_imputada.loc[
            mask_fecha
        ]
        - df_completo.loc[
            mask_fecha,
            "fecha_registro"
        ]
    )
    .abs()
    .dt.days
)

print(
    "MAE temporal:",
    round(
        error_dias.mean(),
        2
    ),
    "días"
)



# 27. Series temporales: ffill, bfill e interpolación

Estas técnicas requieren un **orden temporal real**.


In [ ]:

serie = pd.DataFrame({
    "fecha": pd.date_range(
        "2026-01-01",
        periods=12,
        freq="D"
    ),
    "ventas": [
        100, 110, np.nan, np.nan,
        150, 145, np.nan, 170,
        180, np.nan, 200, 210
    ]
})

serie["ffill"] = (
    serie["ventas"]
    .ffill()
)

serie["bfill"] = (
    serie["ventas"]
    .bfill()
)

serie["interpolacion"] = (
    serie["ventas"]
    .interpolate(
        method="linear"
    )
)

display(serie)


In [ ]:

plt.figure(figsize=(10,5))

plt.plot(
    serie["fecha"],
    serie["ventas"],
    marker="o",
    label="Original"
)

plt.plot(
    serie["fecha"],
    serie["interpolacion"],
    marker="o",
    label="Interpolación"
)

plt.ylabel("Ventas")
plt.title("Interpolación de una serie temporal")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



# 28. Data leakage

Un error grave sería:

1. imputar usando todo el dataset;
2. dividir después en Train/Test.

Así el Test influye en los valores utilizados para imputar Train.

### Solución

Utilizar un `Pipeline`.


# 29. Pipeline para datos mixtos

In [ ]:

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)
from sklearn.linear_model import LogisticRegression

df_ml = df_missing.copy()

df_ml["anio_registro"] = (
    df_ml["fecha_registro"]
    .dt.year
)

df_ml["mes_registro"] = (
    df_ml["fecha_registro"]
    .dt.month
)

df_ml = df_ml.drop(
    columns=["fecha_registro"]
)

X_ml = df_ml.drop(
    columns=["churn"]
)

y_ml = df_ml["churn"]

num_ml = [
    "edad",
    "ingreso",
    "antiguedad",
    "compras",
    "satisfaccion",
    "anio_registro",
    "mes_registro"
]

cat_ml = [
    "ciudad",
    "segmento",
    "canal"
]

bool_ml = [
    "activo"
]


In [ ]:

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]),
        num_ml
    ),
    (
        "cat",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        cat_ml
    ),
    (
        "bool",
        SimpleImputer(
            strategy="most_frequent"
        ),
        bool_ml
    )
])

pipeline_ml = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    )
])


# 30. Validación cruzada del Pipeline

In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

r = cross_validate(
    pipeline_ml,
    X_ml,
    y_ml,
    cv=cv,
    scoring={
        "Accuracy": "accuracy",
        "F1": "f1",
        "ROC_AUC": "roc_auc"
    },
    n_jobs=-1
)

resumen_ml = pd.DataFrame({
    "Accuracy": r["test_Accuracy"],
    "F1": r["test_F1"],
    "ROC_AUC": r["test_ROC_AUC"]
})

display(
    resumen_ml.round(4)
)

display(
    resumen_ml
    .mean()
    .to_frame("Promedio")
    .T
    .round(4)
)



# 31. Comparar imputadores por rendimiento downstream

Un método puede reconstruir mejor los valores, pero no necesariamente producir el mejor modelo final.

Compararemos:

- Media
- Mediana
- KNN
- Iterative


In [ ]:

estrategias_num = {
    "Media": SimpleImputer(
        strategy="mean",
        add_indicator=True
    ),
    "Mediana": SimpleImputer(
        strategy="median",
        add_indicator=True
    ),
    "KNN": KNNImputer(
        n_neighbors=5,
        weights="distance",
        add_indicator=True
    ),
    "Iterative": IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=RANDOM_STATE,
        initial_strategy="median",
        add_indicator=True
    )
}

resultados_downstream = []

for nombre, imputador in estrategias_num.items():

    prep = ColumnTransformer([
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    imputador
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            num_ml
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            cat_ml
        ),
        (
            "bool",
            SimpleImputer(
                strategy="most_frequent"
            ),
            bool_ml
        )
    ])

    pipe = Pipeline([
        (
            "prep",
            prep
        ),
        (
            "modelo",
            LogisticRegression(
                max_iter=3000,
                random_state=RANDOM_STATE
            )
        )
    ])

    cv_temp = cross_validate(
        pipe,
        X_ml,
        y_ml,
        cv=3,
        scoring={
            "F1": "f1",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    resultados_downstream.append({
        "Método": nombre,
        "F1_CV": (
            cv_temp["test_F1"]
            .mean()
        ),
        "ROC_AUC_CV": (
            cv_temp["test_ROC_AUC"]
            .mean()
        )
    })

tabla_downstream = pd.DataFrame(
    resultados_downstream
).sort_values(
    "ROC_AUC_CV",
    ascending=False
)

display(
    tabla_downstream.round(4)
)



# 32. ¿Cómo escoger el mejor método?

Debemos combinar varias perspectivas.

## Reconstrucción
- MAE
- RMSE
- NRMSE

## Distribución
- media;
- mediana;
- dispersión;
- histogramas.

## Estructura multivariada
- correlaciones;
- relaciones entre variables.

## Desempeño final
- clasificación: F1, ROC-AUC, Recall;
- regresión: MAE, RMSE, R².

## Simplicidad
Un método simple puede ser preferible si ofrece rendimiento similar y es más fácil de explicar.



# 33. Guía rápida

| Situación | Método candidato |
|---|---|
| Numérica simétrica | Media |
| Numérica sesgada/outliers | Mediana |
| Observaciones similares | KNN |
| Relaciones multivariadas | Iterative/MICE |
| Categoría con pocos faltantes | Moda |
| Ausencia informativa | DESCONOCIDO + indicador |
| Grupos claramente definidos | Mediana/moda por grupo |
| Serie temporal | Interpolación / ffill |
| Patrón complejo | Modelo predictivo |



# 34. Cuándo puede ser mejor NO imputar

- porcentaje extremo de faltantes;
- variable poco útil;
- ausencia demasiado difícil de justificar;
- riesgo alto de introducir sesgo;
- algoritmo que maneja NaN de forma nativa.

Eliminar una columna también puede ser una decisión válida.



# 35. Modelos que pueden manejar faltantes

Algunos algoritmos tienen soporte nativo para `NaN` en determinados escenarios, por ejemplo:

- `HistGradientBoostingClassifier`
- `HistGradientBoostingRegressor`
- XGBoost
- LightGBM
- CatBoost

Aun así, el mecanismo de ausencia debe analizarse.



# 36. Imputación múltiple

La imputación simple genera un único valor.

La **imputación múltiple** genera varias versiones plausibles del dataset y permite representar mejor la incertidumbre.

Es especialmente relevante en análisis estadístico inferencial.

MICE está relacionado con este enfoque.



# 37. Errores comunes

1. Imputar antes de Train/Test.
2. Usar la media para todas las variables.
3. No revisar outliers.
4. Usar KNN sin controlar escalas.
5. Convertir categorías a números arbitrarios.
6. Imputar el target sin justificación.
7. Usar ffill sin orden temporal.
8. No medir calidad de reconstrucción.
9. No comparar métodos.
10. Confundir el valor imputado con el valor real.
11. Ignorar que la ausencia puede ser informativa.
12. Elegir un método solo porque es más complejo.



# 38. Flujo recomendado

**Diagnóstico → mecanismo de ausencia → tipo de variable → baseline simple → método avanzado → métricas de reconstrucción → distribuciones → Pipeline → validación cruzada → rendimiento downstream → decisión**



# 39. Checklist

Antes de imputar:

- ¿qué porcentaje falta?
- ¿en qué columnas?
- ¿hay patrones?
- ¿MCAR, MAR o posible MNAR?
- ¿tipo de variable?
- ¿hay outliers?
- ¿existe estructura temporal?
- ¿la ausencia tiene significado?
- ¿qué modelo se utilizará después?
- ¿la imputación altera demasiado la distribución?
- ¿qué método funciona mejor en validación?



# 40. Conclusiones

Imputar correctamente requiere mucho más que eliminar `NaN`.

Debemos:

1. entender la ausencia;
2. respetar el tipo de dato;
3. comparar métodos;
4. medir error;
5. observar distribuciones;
6. evitar data leakage;
7. evaluar el impacto en el modelo final.

> **El mejor imputador es el que produce una representación razonable de los datos y funciona bien para el objetivo del proyecto, no necesariamente el más sofisticado.**



# 41. Referencias

- Scikit-learn — Imputation  
  https://scikit-learn.org/stable/modules/impute.html

- SimpleImputer  
  https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html

- KNNImputer  
  https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html

- IterativeImputer  
  https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html

- MissingIndicator  
  https://scikit-learn.org/stable/modules/generated/sklearn.impute.MissingIndicator.html

- Pipeline  
  https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

- ColumnTransformer  
  https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
